# MMELON Model Inference

Run inference with a finetuned MMELON model on **any CSV file** containing SMILES strings.

All dataset, dataloader and task logic lives in **`mmelon_utils.py`** (same folder).

## Steps
0. Configuration — edit paths here
1. Preview input data
2. Load model
3. Build DataLoader
4. Run inference
5. Inspect predictions
6. Evaluate (if labels available)
7. Enrichment metrics (Precision@K / EF@K)
8. Top hits

In [ ]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"Running in Colab: {IN_COLAB}")   

In [ ]:
## installation in local env
# conda create -n mmelon311 python=3.11 -y
# conda activate mmelon311
# git clone https://github.com/jmorrone/biomed-multi-view.git
# pip install git+https://github.com/jmorrone/biomed-multi-view.git
# pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.1.0 torchvision==0.16.0
# pip install --index-url https://download.pytorch.org/whl/cu121 "torch==2.1.0+cu121" "torchvision==0.16.0+cu121" --upgrade-strategy only-if-needed
# pip install -f https://data.pyg.org/whl/torch-2.1.0+cu121.html "pyg_lib==0.4.0+pt21cu121" "torch_scatter==2.1.2+pt21cu121" "torch_cluster==1.6.3+pt21cu121" "torch_spline_conv==1.2.2+pt21cu121" --upgrade-strategy only-if-needed
# pip install -q notebook ipykernel

In [ ]:
if IN_COLAB:
    !git clone https://github.com/jmorrone/biomed-multi-view.git

In [ ]:
if IN_COLAB:
    !pip install git+https://github.com/jmorrone/biomed-multi-view.git  

In [ ]:
if IN_COLAB:
    import os
    os.chdir('biomed-multi-view')
    !pip install -r requirements.txt

## 0. Configuration

**Edit the paths below before running.**

In [ ]:
import torch
run_on_wdr91 = True

if run_on_wdr91:
    # MODEL_PATH = "/proj/ligand_ai/models/wdr91_ASMS_train_val_v1/mmelon_10_seed1234/last-v3.ckpt"
    MODEL_PATH = "michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.wdr91_asms"
    if IN_COLAB:
        DATA_PATH = "/content/wdr91_test.csv"
        OUTPUT_PATH = f"/content/results/mmelon_notebook_predictions_wdr91.csv"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1/test.csv"
        USER = os.getenv("USER")
        OUTPUT_PATH = f"/proj/ligand_ai/users/{USER}/wdr91_mmelon_predictions.csv"
else:
    # MODEL_PATH = "/proj/ligand_ai/models/PGK2_DEL_cdd_to_creative/mmelon_50/last.ckpt"
    MODEL_PATH="michalozeryflato/biomed.sm.mv-te-84m.pgk2_del_cdd"
    if IN_COLAB:
        DATA_PATH = "/content/del_creative.csv"
        OUTPUT_PATH = f"/content/results/pgk2_mmelon_predictions.csv"
    else:
        DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/PGK2_DEL_cdd_to_creative/test.csv"
        USER = os.getenv("USER")
        OUTPUT_PATH = f"/proj/ligand_ai/users/{USER}/pgk2_mmelon_predictions.csv"

# Column names
SMILES_COLUMN = "smiles"
LABEL_COLUMN = "label"

# Inference settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model : {MODEL_PATH}")
print(f"Data  : {DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Device: {DEVICE}")

## 1. Preview Input Data

In [21]:
import pandas as pd

input_df = pd.read_csv(DATA_PATH)
input_df["sample_id"] = input_df.index.astype(str)  # Add sample_id column based on index
display(input_df)

## 2. Load Model

In [4]:
# from mmelon_utils import load_model

# model = load_model(MODEL_PATH, device=DEVICE)
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule
from huggingface_hub import hf_hub_download
checkpoint_path = hf_hub_download(
            repo_id=MODEL_PATH,
            filename="last.ckpt",
            revision="main",
        )
model = FineTuneLightningModule.load_from_checkpoint(
        checkpoint_path,
        map_location=DEVICE
    )
model.to(DEVICE)
model.eval()

## 3. Build DataLoader

In [ ]:
# from mmelon_utils import create_dataloader
import os
import shutil
from torch.utils.data import DataLoader
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.core.data_modules.namespace import TaskType

def create_dataloader(
    csv_path: str,
    smiles_column: str = 'smiles',
    label_column: str = 'label',
    batch_size: int = 32,
    num_workers: int = 0,  # Set to 0 to avoid multiprocessing issues in notebooks
    data_dir: str = None,
):
    print(f"Creating DataLoader from: {csv_path}")
    
    # Get the directory containing the CSV file
    data_dir = data_dir or os.path.dirname(csv_path)
    predict_csv = os.path.join(data_dir, 'data_predict.csv')
    shutil.copy(csv_path, predict_csv)

    # Dataset arguments for MMELON framework (matching evaluate_mmelon.py lines 448-455)
    dataset_args = {
        'task_type': TaskType.CLASSIFICATION,
        'num_tasks': 1,
        'modalities': ['TEXT_MODEL', 'IMAGE_MODEL', 'GRAPH_2D_MODEL'],
        'smiles_col': smiles_column,
        'label_cols': [label_column],
        'split_col': None
    }
    
    # Create dataset (matching evaluate_mmelon.py lines 458-462)
    dataset = MultiModalFinetuneDataPipeline(
        data_dir=data_dir,
        dataset_args=dataset_args,
        stage='predict'  # Use 'predict' stage like evaluate_mmelon.py
    )
    
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=dataset.collate_fn
    )
        
    return dataloader

dataloader = create_dataloader(
    csv_path=DATA_PATH,
    smiles_column=SMILES_COLUMN,
    label_column=LABEL_COLUMN,
    batch_size=32,
    num_workers=4,
    data_dir="/proj/ligand_ai/users/ozery/"
)

In [12]:
first_batch = next(iter(dataloader))
for k,v in first_batch.items():
    print(f"{k}: {type(v)=} {len(v)=}")

## 4. Run Inference

In [36]:
import numpy as np
from tqdm import tqdm
def run_inference(
    model,
    dataloader: DataLoader,
    device: str = 'cuda',
) -> pd.DataFrame:
    
    all_preds = []
    all_probs = []
    
    model.eval()
    batch_count = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="batches"):
            # Move batch to device
            for key in batch:
                if isinstance(batch[key], torch.Tensor):
                    batch[key] = batch[key].to(device)
            
            # Forward pass - use forward0 to get embeddings (same as evaluate_mmelon.py line 486)
            embeddings = model.model.forward0(batch)
            if isinstance(embeddings, tuple):
                embeddings = embeddings[0]  # Extract embeddings from tuple
            
            # Apply prediction head (same as evaluate_mmelon.py line 491)
            logits = model.pred_head(embeddings)
            
            # Squeeze extra dimensions if present
            if logits.dim() > 2:
                logits = logits.squeeze(-1)
            
           
            # Get probabilities - use sigmoid for binary classification (line 494)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.append(probs)
            all_preds.append(preds)
            batch_count += 1
    
    # Concatenate and flatten (same as evaluate_mmelon.py lines 500-501)
    probabilities = np.vstack(all_probs).flatten()
    predictions = np.vstack(all_preds).flatten()
    
    return pd.DataFrame({"predicted_label": predictions, "prediction_score": probabilities})


In [37]:
predictions_df = run_inference(model, dataloader, device=DEVICE)
display(predictions_df)

In [25]:
predictions_df.to_csv(OUTPUT_PATH, index=False)
print("wrote", OUTPUT_PATH)

## 5. Inspect Predictions

In [ ]:
predictions_df = pd.concat([input_df, predictions_df], axis=1)

In [ ]:
predictions_df = predictions_df.rename(columns={LABEL_COLUMN: "true_label" })
display(predictions_df)

In [ ]:
print(f"Shape: {predictions_df.shape}")
print(f"\nPredicted label distribution:")
print(predictions_df["predicted_label"].value_counts().sort_index())
print(f"\nScore statistics:")
print(predictions_df["prediction_score"].describe())

## 6. Evaluate (if labels available)

In [29]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

has_labels = predictions_df["true_label"].notna().any()

if has_labels:
    valid   = predictions_df[predictions_df["true_label"].notna()].copy()
    y_true  = valid["true_label"].astype(int)
    y_pred  = valid["predicted_label"].astype(int)
    y_score = valid["prediction_score"].astype(float)

    print("=" * 50)
    print("CLASSIFICATION METRICS")
    print("=" * 50)
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1 Score  : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC AUC   : {roc_auc_score(y_true, y_score):.4f}")
    print(f"PR AUC    : {average_precision_score(y_true, y_score):.4f}")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No true labels found — skipping evaluation.")

## 7. Enrichment Metrics (Precision@K / EF@K)

In [31]:
def calculate_enrichment_metrics(
    y_true,
    y_scores,
    top_k_values: list[int] | None = None,
) -> dict:
    if top_k_values is None:
        top_k_values = [10, 50, 100, 500, 1000, 2000]

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    sorted_indices = np.argsort(y_scores)[::-1]
    sorted_labels = y_true[sorted_indices]

    hit_prevalence = np.sum(y_true) / len(y_true)

    enrichment: dict = {}
    for k in top_k_values:
        if k <= len(sorted_labels):
            top_k_labels = sorted_labels[:k]
            precision_at_k = np.sum(top_k_labels) / k
            enrichment[f"Precision@{k}"] = precision_at_k
            enrichment[f"EF@{k}"] = (
                precision_at_k / hit_prevalence if hit_prevalence > 0 else np.nan
            )
        else:
            enrichment[f"Precision@{k}"] = np.nan
            enrichment[f"EF@{k}"] = np.nan

    return enrichment

In [32]:
import numpy as np

n_total = len(valid)
top_k_values = sorted({k for k in [10, 50, 100, 500, 1000, 2000] if k <= n_total})
if n_total not in top_k_values:
    top_k_values.append(n_total)

enrichment = calculate_enrichment_metrics(
    y_true.values, y_score.values, top_k_values=top_k_values
)

n_positives = int(y_true.sum())
hit_prev    = n_positives / n_total

print("=" * 60)
print("ENRICHMENT METRICS")
print("=" * 60)
print(f"Dataset : {n_total} compounds  |  Positives: {n_positives}  |  Hit rate: {hit_prev:.2%}")
print()
print(f"{'K':>8}  {'Precision@K':>12}  {'EF@K':>8}  {'Hits in top-K':>14}")
print("-" * 50)
for k in top_k_values:
    prec = enrichment.get(f"Precision@{k}", np.nan)
    ef   = enrichment.get(f"EF@{k}",        np.nan)
    hits = int(round(prec * k)) if not np.isnan(prec) else "N/A"
    print(f"{k:>8}  {prec:>12.4f}  {ef:>8.2f}  {hits:>14}")
print("=" * 60)

ef_rows = [
    {
        "K":             k,
        "Precision@K":   enrichment.get(f"Precision@{k}", np.nan),
        "EF@K":          enrichment.get(f"EF@{k}",        np.nan),
        "Hits_in_top_K": int(round(enrichment.get(f"Precision@{k}", 0) * k)),
    }
    for k in top_k_values
]
ef_df = pd.DataFrame(ef_rows)
ef_df


## 8. Top Hits

In [35]:
top_k = 20
top_predictions_df = (
    predictions_df
    .sort_values("prediction_score", ascending=False)
    .head(top_k)
    .reset_index(drop=True)
)

n_pos = (top_predictions_df["true_label"] == 1).sum()
print(f"Top {top_k} predicted positives (out of {n_pos} total predicted positives):")
display(top_predictions_df[["sample_id", SMILES_COLUMN, "prediction_score", "true_label"]])